# MMA8452Q Accelerometer Tutorial for Beginners

This tutorial provides a comprehensive guide to understanding and using the MMA8452Q accelerometer library for STM32F429 microcontrollers. We'll cover everything from basic concepts to advanced configuration.

## Table of Contents
1. [Introduction to Accelerometers](#introduction)
2. [MMA8452Q Sensor Overview](#sensor-overview)
3. [Hardware Setup](#hardware-setup)
4. [Software Dependencies](#software-setup)
5. [Basic Initialization](#basic-init)
6. [Reading Acceleration Data](#reading-data)
7. [Configuration Options](#configuration)
8. [Calibration](#calibration)
9. [Interrupt System](#interrupts)
10. [Advanced Features](#advanced)
11. [Troubleshooting](#troubleshooting)

<a id='introduction'></a>
## 1. Introduction to Accelerometers

An accelerometer is a sensor that measures acceleration forces. These forces can be:
- **Static**: Gravity (when device is stationary)
- **Dynamic**: Movement, vibration, or shock

### Key Concepts:
- **Acceleration**: Rate of change of velocity (m/s²)
- **G-Force**: Acceleration relative to gravity (1g = 9.81 m/s²)
- **Axes**: X, Y, Z (3D measurement)
- **Range**: Maximum measurable acceleration (±2g, ±4g, ±8g)
- **Resolution**: Precision of measurements

### Applications:
- Motion detection
- Tilt sensing
- Vibration monitoring
- Free-fall detection
- Step counting
- Orientation detection

<a id='sensor-overview'></a>
## 2. MMA8452Q Sensor Overview

The MMA8452Q is a low-power, three-axis accelerometer from NXP Semiconductors.

### Key Specifications:
- **Measurement Range**: ±2g, ±4g, ±8g
- **Resolution**: 14-bit
- **Output Data Rates**: 1.56Hz to 800Hz
- **Interface**: SPI (used in this driver)
- **Supply Voltage**: 1.95V to 3.6V
- **Current Consumption**: ~165µA active, ~6µA standby
- **Device ID**: 0x2A

### Sensor Features:
- 3-axis digital accelerometer
- Embedded interrupt functions
- High-pass filter
- Low-noise mode
- Self-test capability
- FIFO buffer (32 samples)
- Motion detection (freefall, motion, tap)

### Data Format:
- **Raw Data**: 14-bit signed integers (-8192 to +8191)
- **Converted Data**: Floating-point g-force values
- **Sensitivity**: 4096 counts/g (±2g), 2048 counts/g (±4g), 1024 counts/g (±8g)

<a id='hardware-setup'></a>
## 3. Hardware Setup

### Required Components:
- STM32F429 Discovery board
- MMA8452Q accelerometer module
- Connecting wires
- Power supply (3.3V)

### Pin Connections (SPI Interface):

| MMA8452Q Pin | STM32F429 Pin | Description |
|-------------|---------------|-------------|
| VCC | 3.3V | Power supply |
| GND | GND | Ground |
| SCL/SCK | PF7 (SPI5_SCK) | SPI clock |
| SDA/SDI | PF9 (SPI5_MOSI) | SPI MOSI |
| SDO | PF8 (SPI5_MISO) | SPI MISO |
| CS | GPIO Output | Chip select (software controlled) |
| INT1/INT2 | GPIO Input | Interrupt pins (optional) |

### SPI Configuration:
- **Mode**: SPI Mode 0 (CPOL=0, CPHA=0)
- **Data Size**: 8-bit
- **Clock Speed**: ≤1MHz (MMA8452Q limit)
- **Bit Order**: MSB first

### Important Notes:
- Ensure proper power supply (1.95V-3.6V)
- Use decoupling capacitors (10µF + 0.1µF)
- Keep SPI traces short
- Connect interrupt pins if using interrupts

<a id='software-setup'></a>
## 4. Software Dependencies

### Required Files:
- `accel.h` - Header file with function prototypes
- `accel.c` - Main implementation
- `accel_constants.h` - Constants and definitions
- SPI driver (`spi.h`, `spi.c`)

### Library Dependencies:
- STM32F4xx HAL Library
- Standard C libraries: `stdint.h`, `stdbool.h`, `string.h`, `math.h`

### Include Path Setup:
```c
#include "accel.h"
#include "spi.h"  // SPI driver
```

### Build Configuration:
- Add source files to project
- Include header paths
- Link with HAL library
- Enable SPI peripheral in STM32CubeMX

<a id='basic-init'></a>
## 5. Basic Initialization

### Default Initialization:
```c
#include "accel.h"

int main(void) {
    // Initialize accelerometer with default settings
    // - Data rate: 100Hz
    // - Range: ±2g
    // - Mode: Active
    // - High-pass filter: Disabled
    // - Low noise: Disabled
    ACCEL_StatusTypeDef status = ACCEL_Init();
    
    if (status != ACCEL_OK) {
        // Handle initialization error
        printf("Accelerometer initialization failed!\n");
        return -1;
    }
    
    printf("Accelerometer initialized successfully\n");
    
    // Main application loop
    while (1) {
        // Application code here
    }
}
```

### Custom Initialization:
```c
ACCEL_ConfigTypeDef config = {
    .DataRate = ACCEL_ODR_400HZ,      // 400 Hz output rate
    .Range = ACCEL_RANGE_4G,          // ±4g measurement range
    .Mode = ACCEL_MODE_ACTIVE,        // Active mode
    .HighPassFilter = true,           // Enable high-pass filter
    .LowNoise = true                  // Enable low noise mode
};

ACCEL_StatusTypeDef status = ACCEL_Init_Custom(&config);
```

### Initialization Steps:
1. **Device Check**: Verify communication and device ID
2. **Standby Mode**: Put device in standby for configuration
3. **Register Setup**: Configure data rate, range, filters
4. **Active Mode**: Switch to active mode for measurements
5. **Verification**: Confirm settings are applied correctly

<a id='reading-data'></a>
## 6. Reading Acceleration Data

### Processed Data Reading (Recommended):
```c
#include "accel.h"

ACCEL_DataTypeDef data;
ACCEL_StatusTypeDef status = ACCEL_ReadData(&data);

if (status == ACCEL_OK) {
    printf("Acceleration:\n");
    printf("X: %.3f g\n", data.X_g);
    printf("Y: %.3f g\n", data.Y_g);
    printf("Z: %.3f g\n", data.Z_g);
    
    // Raw values are also available
    printf("Raw X: %d\n", data.X);
    printf("Raw Y: %d\n", data.Y);
    printf("Raw Z: %d\n", data.Z);
}
```

### Raw Data Reading:
```c
int16_t xRaw, yRaw, zRaw;
ACCEL_StatusTypeDef status = ACCEL_ReadRawData(&xRaw, &yRaw, &zRaw);

if (status == ACCEL_OK) {
    // Raw values are 14-bit signed (-8192 to +8191)
    printf("Raw: X=%d, Y=%d, Z=%d\n", xRaw, yRaw, zRaw);
    
    // Convert to g-force manually if needed
    float x_g = ACCEL_ConvertToG(xRaw, ACCEL_RANGE_2G);
    float y_g = ACCEL_ConvertToG(yRaw, ACCEL_RANGE_2G);
    float z_g = ACCEL_ConvertToG(zRaw, ACCEL_RANGE_2G);
}
```

### Data Conversion:
```c
// Manual conversion function
float convertToG(int16_t raw, uint8_t range) {
    float sensitivity;
    
    switch (range) {
        case ACCEL_RANGE_2G: sensitivity = 4096.0f; break;
        case ACCEL_RANGE_4G: sensitivity = 2048.0f; break;
        case ACCEL_RANGE_8G: sensitivity = 1024.0f; break;
        default: sensitivity = 4096.0f; break;
    }
    
    return (float)raw / sensitivity;
}
```

### Understanding the Data:
- **Stationary device**: Z-axis should read ~1.0g (gravity)
- **X/Y axes**: Should read ~0.0g when level
- **Movement**: Values change based on acceleration
- **Free-fall**: All axes read ~0.0g
- **Shock**: Sudden spikes in values

<a id='configuration'></a>
## 7. Configuration Options

### Output Data Rates:
```c
// Available data rates
ACCEL_SetDataRate(ACCEL_ODR_800HZ);   // 800 Hz - Fastest
ACCEL_SetDataRate(ACCEL_ODR_400HZ);   // 400 Hz
ACCEL_SetDataRate(ACCEL_ODR_200HZ);   // 200 Hz
ACCEL_SetDataRate(ACCEL_ODR_100HZ);   // 100 Hz - Default
ACCEL_SetDataRate(ACCEL_ODR_50HZ);    // 50 Hz
ACCEL_SetDataRate(ACCEL_ODR_12_5HZ);  // 12.5 Hz
ACCEL_SetDataRate(ACCEL_ODR_6_25HZ);  // 6.25 Hz
ACCEL_SetDataRate(ACCEL_ODR_1_56HZ);  // 1.56 Hz - Slowest
```

### Measurement Ranges:
```c
// Available ranges
ACCEL_SetRange(ACCEL_RANGE_2G);  // ±2g - Highest sensitivity
ACCEL_SetRange(ACCEL_RANGE_4G);  // ±4g - Medium sensitivity
ACCEL_SetRange(ACCEL_RANGE_8G);  // ±8g - Lowest sensitivity
```

### Operating Modes:
```c
ACCEL_SetMode(ACCEL_MODE_ACTIVE);   // Normal operation
ACCEL_SetMode(ACCEL_MODE_STANDBY);  // Low power, configuration
ACCEL_SetMode(ACCEL_MODE_SLEEP);    // Ultra low power
```

### Filter Options:
```c
// High-pass filter (removes DC offset/gravity)
ACCEL_EnableHighPassFilter(true);   // Enable
ACCEL_EnableHighPassFilter(false);  // Disable

// Low noise mode (improves accuracy)
ACCEL_EnableLowNoise(true);         // Enable
ACCEL_EnableLowNoise(false);        // Disable
```

### Configuration Trade-offs:
- **Higher data rates**: More responsive, higher power consumption
- **Larger ranges**: Can measure higher accelerations, lower resolution
- **High-pass filter**: Removes gravity component, shows dynamic acceleration only
- **Low noise mode**: Better accuracy, slightly higher power consumption

<a id='calibration'></a>
## 8. Calibration

### Automatic Calibration:
```c
// Keep sensor level and stationary during calibration
ACCEL_StatusTypeDef status = ACCEL_Calibrate();

if (status == ACCEL_OK) {
    printf("Calibration completed successfully\n");
} else {
    printf("Calibration failed\n");
}
```

### Manual Offset Adjustment:
```c
// Set manual offset values (-128 to +127)
ACCEL_SetOffset(10, -5, 20);  // X, Y, Z offsets
```

### Reading Current Offsets:
```c
int8_t xOffset, yOffset, zOffset;
ACCEL_GetOffset(&xOffset, &yOffset, &zOffset);

printf("Current offsets: X=%d, Y=%d, Z=%d\n", 
       xOffset, yOffset, zOffset);
```

### Calibration Procedure:
1. Place sensor on level surface
2. Ensure sensor is stationary
3. Call `ACCEL_Calibrate()`
4. Verify readings: X=0, Y=0, Z=1.0g

### When to Calibrate:
- After initial setup
- When accuracy seems off
- After significant temperature changes
- If sensor has been moved/shocked

### Calibration Tips:
- Use automatic calibration when possible
- Keep sensor level during calibration
- Avoid vibrations during calibration
- Calibrate in the target environment

<a id='interrupts'></a>
## 9. Interrupt System

### Interrupt Configuration:
```c
ACCEL_IntConfigTypeDef intConfig = {
    .DataReady = true,     // Interrupt on new data
    .Motion = false,       // Motion detection disabled
    .Freefall = false,     // Freefall detection disabled
    .Tap = true            // Tap detection enabled
};

ACCEL_StatusTypeDef status = ACCEL_ConfigInterrupts(&intConfig);
```

### Interrupt Sources:
- **Data Ready**: New acceleration data available
- **Motion**: Acceleration exceeds threshold
- **Freefall**: All axes near zero (freefall detected)
- **Tap**: Single or double tap detected

### Reading Interrupt Status:
```c
uint8_t intSource;
ACCEL_GetInterruptSource(&intSource);

if (intSource & 0x01) printf("Data Ready\n");
if (intSource & 0x04) printf("Motion Detected\n");
if (intSource & 0x08) printf("Freefall Detected\n");
if (intSource & 0x10) printf("Tap Detected\n");
```

### Hardware Setup for Interrupts:
```c
// Configure GPIO for interrupt input
GPIO_InitTypeDef GPIO_InitStruct = {
    .Pin = GPIO_PIN_0,              // INT1 pin
    .Mode = GPIO_MODE_IT_RISING,    // Rising edge
    .Pull = GPIO_NOPULL
};
HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);

// Enable interrupt
HAL_NVIC_SetPriority(EXTI0_IRQn, 0, 0);
HAL_NVIC_EnableIRQ(EXTI0_IRQn);
```

### Interrupt Handler:
```c
void EXTI0_IRQHandler(void) {
    HAL_GPIO_EXTI_IRQHandler(GPIO_PIN_0);
}

void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == GPIO_PIN_0) {
        // Handle accelerometer interrupt
        uint8_t intSource;
        ACCEL_GetInterruptSource(&intSource);
        
        if (intSource & 0x01) {
            // Data ready interrupt
            ACCEL_DataTypeDef data;
            ACCEL_ReadData(&data);
            // Process data
        }
    }
}
```

<a id='advanced'></a>
## 10. Advanced Features

### Self-Test Function:
```c
// Run built-in self-test
ACCEL_StatusTypeDef status = ACCEL_SelfTest();

if (status == ACCEL_OK) {
    printf("Self-test passed\n");
} else {
    printf("Self-test failed\n");
}
```

### Device Status Checking:
```c
// Check if device is ready
ACCEL_StatusTypeDef status = ACCEL_IsReady();

// Get device ID
uint8_t deviceId;
ACCEL_GetDeviceID(&deviceId);
printf("Device ID: 0x%02X (expected: 0x2A)\n", deviceId);

// Get current mode
uint8_t mode;
ACCEL_GetMode(&mode);

// Get current range
uint8_t range;
ACCEL_GetRange(&range);
```

### FIFO Buffer (Advanced):
The MMA8452Q has a 32-sample FIFO buffer for advanced applications:
- Reduces interrupt frequency
- Allows burst reading
- Helps with data synchronization

### Motion Detection Setup:
```c
// Configure motion detection thresholds
// This requires direct register access for advanced settings
// Refer to MMA8452Q datasheet for details
```

### Power Management:
- **Active Mode**: Normal operation (~165µA)
- **Standby Mode**: Configuration mode (~6µA)
- **Sleep Mode**: Ultra-low power with auto-wake

### Temperature Considerations:
- Operating range: -40°C to +85°C
- Calibration may drift with temperature
- Re-calibrate if temperature changes significantly

<a id='troubleshooting'></a>
## 11. Troubleshooting

### Common Issues and Solutions:

#### 1. Initialization Fails
**Symptoms:** `ACCEL_Init()` returns error
**Possible Causes:**
- SPI communication not working
- Wrong pin connections
- Power supply issues
- Device not powered on
**Solutions:**
- Check SPI wiring and configuration
- Verify power supply (1.95V-3.6V)
- Test with oscilloscope on SPI lines
- Check device ID with `ACCEL_GetDeviceID()`

#### 2. Incorrect Readings
**Symptoms:** Acceleration values don't make sense
**Possible Causes:**
- Wrong measurement range
- Uncalibrated sensor
- Incorrect data conversion
**Solutions:**
- Verify range setting with `ACCEL_GetRange()`
- Perform calibration with `ACCEL_Calibrate()`
- Check data conversion formulas

#### 3. Noisy Data
**Symptoms:** Readings fluctuate when stationary
**Possible Causes:**
- Electrical noise on SPI lines
- Mechanical vibrations
- Low sample rate
**Solutions:**
- Add decoupling capacitors
- Enable low noise mode
- Increase data rate
- Use averaging/filtering

#### 4. Interrupt Issues
**Symptoms:** Interrupts not triggering
**Possible Causes:**
- Wrong GPIO configuration
- Interrupt pins not connected
- Interrupt not enabled in NVIC
**Solutions:**
- Check GPIO and EXTI configuration
- Verify interrupt pin connections
- Enable NVIC interrupt
- Check interrupt source with `ACCEL_GetInterruptSource()`

### Debug Tips:
```c
// Check device communication
uint8_t deviceId;
if (ACCEL_GetDeviceID(&deviceId) == ACCEL_OK) {
    printf("Device ID: 0x%02X\n", deviceId);
} else {
    printf("Cannot communicate with device\n");
}

// Check current configuration
uint8_t mode, range;
ACCEL_GetMode(&mode);
ACCEL_GetRange(&range);
printf("Mode: %d, Range: %d\n", mode, range);

// Monitor raw data
int16_t x, y, z;
ACCEL_ReadRawData(&x, &y, &z);
printf("Raw data: %d, %d, %d\n", x, y, z);
```

### Performance Optimization:
- Use appropriate data rates for your application
- Enable interrupts to avoid polling
- Use FIFO buffer for high-speed applications
- Implement data filtering in software if needed

### Getting Help:
- Check MMA8452Q datasheet for register details
- Verify SPI timing with oscilloscope
- Test with known working hardware
- Review example code implementations

## Complete Example Program

```c
#include "accel.h"
#include "spi.h"
#include <stdio.h>

int main(void) {
    printf("MMA8452Q Accelerometer Demo\n\n");
    
    // Initialize accelerometer
    printf("Initializing accelerometer...\n");
    ACCEL_StatusTypeDef status = ACCEL_Init();
    if (status != ACCEL_OK) {
        printf("Initialization failed: %s\n", ACCEL_GetStatusString(status));
        return -1;
    }
    printf("Initialization successful!\n\n");
    
    // Run self-test
    printf("Running self-test...\n");
    status = ACCEL_SelfTest();
    if (status == ACCEL_OK) {
        printf("Self-test passed!\n\n");
    } else {
        printf("Self-test failed: %s\n\n", ACCEL_GetStatusString(status));
    }
    
    // Calibrate sensor
    printf("Calibrating sensor (keep level and stationary)...\n");
    HAL_Delay(2000); // Give user time to position sensor
    status = ACCEL_Calibrate();
    if (status == ACCEL_OK) {
        printf("Calibration successful!\n\n");
    } else {
        printf("Calibration failed: %s\n\n", ACCEL_GetStatusString(status));
    }
    
    // Main measurement loop
    printf("Starting measurements (Ctrl+C to stop)...\n\n");
    
    while (1) {
        ACCEL_DataTypeDef data;
        status = ACCEL_ReadData(&data);
        
        if (status == ACCEL_OK) {
            printf("\rX: %+6.3f g  Y: %+6.3f g  Z: %+6.3f g", 
                   data.X_g, data.Y_g, data.Z_g);
            fflush(stdout);
        } else {
            printf("\rError reading data: %s", ACCEL_GetStatusString(status));
        }
        
        HAL_Delay(100); // 10 Hz display rate
    }
    
    return 0;
}
```

## Summary

This tutorial covered:
- Accelerometer fundamentals
- MMA8452Q sensor capabilities
- Hardware and software setup
- Basic and advanced usage
- Configuration options
- Calibration procedures
- Interrupt handling
- Troubleshooting techniques

### Key Takeaways:
1. **Always check return values** from API functions
2. **Calibrate regularly** for accurate measurements
3. **Choose appropriate settings** for your application
4. **Use interrupts** to avoid polling overhead
5. **Handle errors gracefully** in production code

### Next Steps:
- Experiment with different configurations
- Implement motion detection algorithms
- Add data filtering and processing
- Integrate with other sensors
- Develop application-specific features

For more detailed information, refer to the MMA8452Q datasheet and the library documentation.